In [17]:
# ─── Fix Java Path First (always run this first!) ────
import os
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

print("✅ Java path set!")

✅ Java path set!


In [18]:
# ─── CELL 1: Set ALL environment variables first ─────
import os

# Java path
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot"

# Hadoop path (winutils)
os.environ["HADOOP_HOME"] = r"C:\hadoop"

# Add both to PATH
os.environ["PATH"] = (
    os.environ["JAVA_HOME"] + r"\bin;" +
    os.environ["HADOOP_HOME"] + r"\bin;" +
    os.environ["PATH"]
)

print("✅ Java home:", os.environ["JAVA_HOME"])
print("✅ Hadoop home:", os.environ["HADOOP_HOME"])

✅ Java home: C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot
✅ Hadoop home: C:\hadoop


In [20]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Test") \
    .getOrCreate()

print("Spark works!")

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

In [19]:
# ─── CELL 2: Start Spark ─────────────────────────────
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Read PostgreSQL") \
    .config(
        "spark.jars",
        r"C:\spark-jars\postgresql-42.7.10.jar"
    ) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("✅ Spark started!")

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

In [ ]:
# ─── CELL 3: PostgreSQL Connection ───────────────────

pg_url = "jdbc:postgresql://localhost:5432/postgres"
#                                           ^^^^^^^^
#                                    change if your DB name is different

pg_properties = {
    "user":     "postgres",   # your username
    "password": "postgres",   # your password
    "driver":   "org.postgresql.Driver"
}

print("✅ Connection details ready!")
print(f"URL: {pg_url}")

✅ Connection details ready!
URL: jdbc:postgresql://localhost:5432/postgres


In [ ]:
# ─── CELL 4: Get All Tables ───────────────────────────

all_tables = spark.read.jdbc(
    url=pg_url,
    table="""(
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        AND table_type = 'BASE TABLE'
        ORDER BY table_name
    ) AS tables""",
    properties=pg_properties
)

table_list = [row["table_name"] for row in all_tables.collect()]

print(f"✅ Found {len(table_list)} tables:\n")
for i, name in enumerate(table_list, 1):
    print(f"   {i}. {name}")

✅ Found 28 tables:

   1. constraint_orders
   2. constraint_products
   3. customers
   4. daily_sales
   5. dim_brand
   6. dim_category
   7. dim_city
   8. dim_customer
   9. dim_customer_snowflake
   10. dim_date
   11. dim_product
   12. dim_product_snowflake
   13. dim_province
   14. dim_store
   15. dim_subcategory
   16. dirty_cafe_sales_dataset
   17. enrollments
   18. fact_sales
   19. fact_sales_snowflake
   20. olist_customers_dataset
   21. order_items
   22. orders
   23. products
   24. students
   25. subjects
   26. teachers
   27. titanic_dataset
   28. windows_function_table


In [ ]:
# ─── CELL 5: Load All Tables ─────────────────────────

all_dataframes = {}

for table_name in table_list:
    print(f"📋 Loading: {table_name} ...", end=" ")
    df = spark.read.jdbc(
        url=pg_url,
        table=table_name,
        properties=pg_properties
    )
    all_dataframes[table_name] = df
    print(f"✅ {df.count()} rows")

print(f"\n✅ All {len(all_dataframes)} tables loaded!")

📋 Loading: constraint_orders ... ✅ 4 rows
📋 Loading: constraint_products ... ✅ 4 rows
📋 Loading: customers ... ✅ 16 rows
📋 Loading: daily_sales ... ✅ 31 rows
📋 Loading: dim_brand ... ✅ 7 rows
📋 Loading: dim_category ... ✅ 2 rows
📋 Loading: dim_city ... ✅ 5 rows
📋 Loading: dim_customer ... ✅ 5 rows
📋 Loading: dim_customer_snowflake ... ✅ 5 rows
📋 Loading: dim_date ... ✅ 10 rows
📋 Loading: dim_product ... ✅ 8 rows
📋 Loading: dim_product_snowflake ... ✅ 8 rows
📋 Loading: dim_province ... ✅ 5 rows
📋 Loading: dim_store ... ✅ 4 rows
📋 Loading: dim_subcategory ... ✅ 6 rows
📋 Loading: dirty_cafe_sales_dataset ... ✅ 10000 rows
📋 Loading: enrollments ... ✅ 8 rows
📋 Loading: fact_sales ... ✅ 12 rows
📋 Loading: fact_sales_snowflake ... ✅ 5 rows
📋 Loading: olist_customers_dataset ... ✅ 198882 rows
📋 Loading: order_items ... ✅ 45 rows
📋 Loading: orders ... ✅ 25 rows
📋 Loading: products ... ✅ 20 rows
📋 Loading: students ... ✅ 6 rows
📋 Loading: subjects ... ✅ 5 rows
📋 Loading: teachers ... ✅ 4 rows
📋 

In [ ]:
# ─── CELL 6: Explore Data ────────────────────────────

# Show all available tables
print("Available tables:", list(all_dataframes.keys()))

# Pick any table and explore
# Change "students" to your actual table name
students_table = all_dataframes["students"]

print("\n📋 Schema:")
table.printSchema()

print("\n📋 First 10 rows:")
table.show(10, truncate=False)

print("\n📋 Total rows:", table.count())

Available tables: ['constraint_orders', 'constraint_products', 'customers', 'daily_sales', 'dim_brand', 'dim_category', 'dim_city', 'dim_customer', 'dim_customer_snowflake', 'dim_date', 'dim_product', 'dim_product_snowflake', 'dim_province', 'dim_store', 'dim_subcategory', 'dirty_cafe_sales_dataset', 'enrollments', 'fact_sales', 'fact_sales_snowflake', 'olist_customers_dataset', 'order_items', 'orders', 'products', 'students', 'subjects', 'teachers', 'titanic_dataset', 'windows_function_table']

📋 Schema:
root
 |-- student_id: integer (nullable = true)
 |-- student_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- is_active: boolean (nullable = true)


📋 First 10 rows:
+----------+---------------+---------+---------+
|student_id|student_name   |city     |is_active|
+----------+---------------+---------+---------+
|1         |Aarav Thapa    |Kathmandu|true     |
|2         |Priya Gurung   |Pokhara  |true     |
|3         |Suman Poudel   |Butwal   |true     |
|4    

In [ ]:
del students_table